In [1]:
from typing import LiteralString, Any
import random
import numpy as np

from autograd import Node

In [2]:
# Three sequential samples (one feature each)
x1 = Node(data=1.0)
x2 = Node(data=2.0)
x3 = Node(data=3.0)

# Input weights: one per hidden neuron
w1 = Node(data=-0.45)
w2 = Node(data=-0.11)

# Hidden layer biases
b11 = Node(data=0.1)
b12 = Node(data=-0.1)

# Initial hidden state: one value per hidden neuron
h0 = np.array([Node(data=0.0), Node(data=0.0)])
# Recurrent weights (2x2)
wh = np.array([
    [Node(data=random.uniform(-1, 1)) for _ in range(2)]
    for _ in range(2)
])
# Output layer weights
wo1 = Node(data=random.uniform(-1, 1))
wo2 = Node(data=random.uniform(-1, 1))

# Output layer bias
bo = Node(data=random.uniform(-1, 1))

### t1

In [3]:
# perspective: t1, h1, n1
t1h1n1 = (
    x1*w1
    + h0[0]*wh[0, 0]
    + h0[1]*wh[0, 1]
    + b11
)

# perspective: t1, h1, n2
t1h1n2 = (
    x1*w2
    + h0[0]*wh[1, 0]
    + h0[1]*wh[1, 1]
    + b12
)
y1 = (t1h1n1*wo1 + t1h1n2*wo2) + bo

### t2

In [4]:
# perspective: t2, h1, n1
t2h1n1 = (
    x2*w1
    + t1h1n1*wh[0, 0]
    + t1h1n2*wh[0, 1]
    + b11
)

# perspective: t2, h1, n2
t2h1n2 = (
    x2*w2
    + t1h1n1*wh[1, 0]
    + t1h1n2*wh[1, 1]
    + b12
)
y2 = (t2h1n1*wo1 + t2h1n2*wo2) + bo

In [5]:
# perspective: t3, h1, n1
t3h1n1 = (
    x3*w1
    + t2h1n1*wh[0, 0]
    + t2h1n2*wh[0, 1]
    + b11
)

# perspective: t3, h1, n2
t3h1n2 = (
    x3*w2
    + t2h1n1*wh[1, 0]
    + t2h1n2*wh[1, 1]
    + b12
)

# perspective: output
y3 = (t3h1n1*wo1 + t3h1n2*wo2) + bo

In [6]:
class Neuron:

    def __init__(self, n: int, weights: list | None = None, bias: float | None = None) -> None:
        self.n_inputs = n
        self.weights = weights if weights else \
            [Node(data=random.uniform(-1, 1)) for _ in range(n)]
        self.bias = bias if bias is not None else \
            Node(random.uniform(-1, 1))

    def _forward_linear(self, X):
        assert len(X) == self.n_inputs
        return sum(
            (xi*wi for xi, wi in zip(X, self.weights)),
            start=self.bias
        )

    def __call__(self, X, activation_fn='relu'):
        return self._forward_linear(X)

    def parameters(self)-> list | list[Node]:
        return self.weights + [self.bias]

class RNNCell(Neuron):

    def __init__(self, 
                 n, 
                 weights = None, 
                 bias = None,
                 hidden_state: float = 0.0,
                 hidden_weight: float | None = None
                 ):
        super().__init__(n, weights, bias)
        self.hidden_state = Node(data=hidden_state)
        self.hidden_weight = Node(data=hidden_weight) if hidden_weight is not None \
            else Node(data=random.uniform(-1, 1))

    def __call__(self, X, activation_fn='relu'):
        activation = (
            self._forward_linear(X)
            + self.hidden_state*self.hidden_weight
        )
        self.hidden_state = activation
        return activation

    def parameters(self):
        return super().parameters() + [self.hidden_weight]

    def reset_state(self):
        self.hidden_state = Node(data=0.0)

class LinearLayer:

    def __init__(self, in_features: int, out_features: int, neuron_cls = Neuron):
        self.in_features = in_features
        self.out_features = out_features
        self.neurons = [neuron_cls(n=self.in_features) for _ in range(self.out_features)]

    def __repr__(self) -> str:
        return f"{type(self).__name__} (in: {self.in_features}, out: {self.out_features})"

    def __call__(self, x: list | np.ndarray):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [param for neuron in self.neurons for param in neuron.parameters()]

class RNNLinearLayer(LinearLayer):

    def __init__(self, in_features, out_features, neuron_cls=Neuron):
        super().__init__(in_features, out_features, neuron_cls)

    def reset_state(self):
        for neuron in self.neurons: neuron.reset_state()

class Network:

    def __init__(self, layers: list, layer_cls = LinearLayer) -> None:
        self.layers = [layer_cls(
            in_features=layers[i], 
            out_features=layers[i+1], 
            neuron_cls=RNNCell) for i in range(len(layers) - 1)]

    def __repr__(self) -> str:
            return f"Net (Layers ({[layer for layer in self.layers]}))"

    def __call__(self, x: list | np.ndarray, final_activation: bool = True):
        def apply_activation(activation):
            return [node.relu() for node in activation]

        for i, layer in enumerate(self.layers):
            x = layer(x)
            # ReLU on hidden layers
            if i != len(self.layers) - 1:
                x = apply_activation(activation=x)
            elif final_activation:
                x = apply_activation(activation=x)
                
        return x[0] if len(x) == 1 else x # if only one feature

    def parameters(self)-> list[Any | Node]:
        return [param for layer in self.layers for param in layer.parameters()]

    def zero_grad(self) -> None:
        for param in self.parameters(): param.grad = 0

    def reset_reccurent_state(self) -> None:
        for layer in self.layers: layer.reset_state()

In [20]:
# synthetic sequential data
X = [
    [[1], [2], [3], [4], [5]],
    [[2], [3], [4], [5], [6]],
    # [[3], [4], [5], [6], [7]],
    # [[4], [5], [6], [7], [8]]
]
y = []
for sample in X:
    sam = []
    for t in sample:
        sam.append(t[0]+1)
    y.append(sam)

# y = [
#     6,
#     7,
#     8,
#     9
# ]
net = Network(layers=[1, 2, 1], layer_cls=RNNLinearLayer)
y

[[2, 3, 4, 5, 6], [3, 4, 5, 6, 7]]

In [21]:
# regression example
lr = 0.001
gradients = []  # capture model Node params for each epoch
# Training
for epoch in range(30):
    sample_predictions = []
    for sample in X:
        timestep_preds = []
        # reset reccurrent states
        net.reset_reccurent_state()

        # Forward pass (for single output)
        for timestep in sample:
            y_hat = net(timestep, final_activation=False)
            timestep_preds.append(y_hat)

        sample_predictions.append(timestep_preds)

    loss = sum(
        (y_true - y_hat) ** 2
        for sample_y, sample_y_hat in zip(y, sample_predictions)
        for y_true, y_hat in zip(sample_y, sample_y_hat)
    ) / sum(len(sample) for sample in y)
    print(f"Epoch {epoch} loss: {loss.data}")

    net.zero_grad()
    # Backpropagation
    loss.backward()

    gradients.extend(
        {f'Epoch {epoch}': {'Param number': i, 'data': param.data, 'gradient': param.grad}}
        for i, param in enumerate(net.parameters())
    )

    # for i, param in enumerate(net.parameters()):
    #     print(
    #         f"Param {i}: "
    #         f"data={param.data:.6f}, "
    #         f"grad={param.grad:.6f}"
    #     )

    # Gradient descent
    for param in net.parameters():
        param.data -= lr * param.grad

    print(f"Epoch {epoch}: Loss = {loss.data:.6f}")

Epoch 0 loss: 0.9710487777845224
Epoch 0: Loss = 0.971049
Epoch 1 loss: 0.8630401223845501
Epoch 1: Loss = 0.863040
Epoch 2 loss: 0.8078091917703599
Epoch 2: Loss = 0.807809
Epoch 3 loss: 0.7771497259123287
Epoch 3: Loss = 0.777150
Epoch 4 loss: 0.7585390014890191
Epoch 4: Loss = 0.758539
Epoch 5 loss: 0.7460566720441681
Epoch 5: Loss = 0.746057
Epoch 6 loss: 0.7367801877227755
Epoch 6: Loss = 0.736780
Epoch 7 loss: 0.7292217817391589
Epoch 7: Loss = 0.729222
Epoch 8 loss: 0.72260683347099
Epoch 8: Loss = 0.722607
Epoch 9 loss: 0.71652513207226
Epoch 9: Loss = 0.716525
Epoch 10 loss: 0.7107565588589972
Epoch 10: Loss = 0.710757
Epoch 11 loss: 0.7051817199760513
Epoch 11: Loss = 0.705182
Epoch 12 loss: 0.6997352421614588
Epoch 12: Loss = 0.699735
Epoch 13 loss: 0.6943810019660455
Epoch 13: Loss = 0.694381
Epoch 14 loss: 0.6890988361007537
Epoch 14: Loss = 0.689099
Epoch 15 loss: 0.6838773462563779
Epoch 15: Loss = 0.683877
Epoch 16 loss: 0.6787099747533085
Epoch 16: Loss = 0.678710
Epoc

In [ ]:
# meant to be called on the last node (ouptut of loss function)
seen = set()
topo = []

def dfs(node) -> None:
    if node not in seen:
        seen.add(node)
        for parent in node.parents: dfs(parent)
        topo.append(node)

dfs(loss)
topo

In [ ]:
for node in topo:
    if 'ops=relu' in str(node): print(node)